In [ ]:
conda activate mandarin-translator-env

In [ ]:
pip uninstall -y pyannote.audio pyannote.core pyannote.metrics
pip install "pyannote.audio>=4.0.1" "numpy>=2.0"

In [ ]:
%pip install funasr modelscope transformers sentencepiece openai torchaudio openpyxl pyannote.audio

In [ ]:
%pip install torch==2.4.0 torchvision==0.18.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

In [5]:
#pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

In [ ]:
#pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu130 --force-reinstall

In [ ]:
#pip install soundfile

In [1]:
# Check if CUDA GPU is available to PyTorch
import torch                                                # PyTorch
torch.cuda.set_device(0)                                    # Set the main GPU as device to use if present
print(torch.__version__)
torch.cuda.is_available(),torch.cuda.get_device_name()      # Check if GPU is available and get the name of the GPU

2.10.0+cu130


(True, 'NVIDIA GeForce RTX 4060 Laptop GPU')

In [2]:
import torch
x = torch.rand(5, 3)
print(x)

tensor([[0.5102, 0.7747, 0.3230],
        [0.7303, 0.5823, 0.9287],
        [0.8910, 0.4135, 0.6867],
        [0.8016, 0.0039, 0.2041],
        [0.6095, 0.4957, 0.6251]])


In [3]:
import torch 
print(f'CUDA Available: {torch.cuda.is_available()}'); print(f'GPU: {torch.cuda.get_device_name(0)}')

CUDA Available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [1]:
import torch

# 1. Safely check for GPU
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Runtime Version: {torch.version.cuda}")
else:
    device = torch.device("cpu")
    print("❌ GPU NOT detected. Running on CPU (will be very slow).")

# 2. Print PyTorch version for your records
print(f"PyTorch Version: {torch.__version__}")

# Use this 'device' variable when loading your models later:
# model.to(device)

✅ GPU Detected: NVIDIA GeForce RTX 4060 Laptop GPU
CUDA Runtime Version: 13.0
PyTorch Version: 2.10.0+cu130


In [4]:
import torch
import torchaudio
import pyannote.audio
from funasr import AutoModel

print(f"--- Environment Check ---")
print(f"✅ PyTorch: {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"✅ TorchAudio: {torchaudio.__version__}")
print(f"✅ Pyannote.audio: {pyannote.audio.__version__}")

# Check if the backend issue is resolved
try:
    from pyannote.audio import Pipeline
    print("✅ Pyannote Import: SUCCESS")
except AttributeError as e:
    print(f"❌ Pyannote Import: FAILED ({e})")

--- Environment Check ---
✅ PyTorch: 2.10.0+cu130 (CUDA: True)
✅ TorchAudio: 2.10.0+cu130
✅ Pyannote.audio: 4.0.4
✅ Pyannote Import: SUCCESS


# Installation & Setup

In [5]:

import torch
import torchaudio
import os
import gc
import pandas as pd
from tqdm import tqdm
from funasr import AutoModel
from pyannote.audio import Pipeline
from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToText
from openai import OpenAI
from tkinter import Tk, filedialog

# --- Configuration ---
HF_TOKEN = os.getenv('HF_TOKEN') 
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY') 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Running on: {DEVICE}")

Running on: cuda


Use instead of one before

In [ ]:
import torch
import torchaudio
from funasr import AutoModel
from pyannote.audio import Pipeline
from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToText
from openai import OpenAI

# 1. Final Verified Configuration
HF_TOKEN = "YOUR_HUGGING_FACE_TOKEN"
OPENAI_API_KEY = "YOUR_OPENAI_API_KEY"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"🚀 Pipeline Initialized on: {DEVICE}")

# 2. Load Models with 2026-Compatible Logic
# SenseVoice for Mandarin + Emotion Tags
transcription_model = AutoModel(model="iic/SenseVoiceSmall", device=str(DEVICE), hub="ms")

# Pyannote 4.0+ (Note the 'token' keyword change)
diarize_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1", 
    token=HF_TOKEN
).to(DEVICE)

# SeamlessM4T v2 for Direct Translation
processor = AutoProcessor.from_pretrained("facebook/seamless-m4t-v2-large")
direct_model = SeamlessM4Tv2ForSpeechToText.from_pretrained("facebook/seamless-m4t-v2-large").to(DEVICE)

# OpenAI for Chained Research Translation
client = OpenAI(api_key=OPENAI_API_KEY)

print("✅ All models loaded and ready for batch processing.")

# Model Initialization

In [ ]:
print("Loading Transcription & Diarization Models...")
# 1. SenseVoice for Mandarin + Emotions
transcription_model = AutoModel(model="iic/SenseVoiceSmall", device=DEVICE, hub="ms")

# 2. Pyannote for Speaker ID
diarize_pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", token=HF_TOKEN).to(torch.device(DEVICE))

# 3. SeamlessM4T for Direct Translation
processor = AutoProcessor.from_pretrained("facebook/seamless-m4t-v2-large")
direct_model = SeamlessM4Tv2ForSpeechToText.from_pretrained("facebook/seamless-m4t-v2-large").to(DEVICE)

# 4. OpenAI for Chained Translation
client = OpenAI(api_key=OPENAI_API_KEY)

Loading Transcription & Diarization Models...
funasr version: 1.3.1.
Check update of funasr, and it would cost few times. You may disable it by set `disable_update=True` in AutoModel
You are using the latest version of funasr-1.3.1


2026-03-28 02:39:50,672 - modelscope - INFO - Got 19 files, start to download ...


Processing 19 items:   0%|          | 0.00/19.0 [00:00<?, ?it/s]

# Processing Functions

In [ ]:
def get_chained_translation(zh_text):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a research assistant. Translate this Mandarin transcript to English. Preserve all emotional tags like <|HAPPY|> or <|LAUGHTER|>."},
            {"role": "user", "content": zh_text}
        ]
    )
    return response.choices[0].message.content

def get_direct_translation(audio_path):
    audio, orig_freq = torchaudio.load(audio_path)
    if orig_freq != 16000:
        audio = torchaudio.functional.resample(audio, orig_freq=orig_freq, new_freq=16000)
    inputs = processor(audios=audio, return_tensors="pt").to(DEVICE)
    output_tokens = direct_model.generate(**inputs, tgt_lang="eng")
    return processor.decode(output_tokens[0].tolist()[0], skip_special_tokens=True)

# Processing Functions

In [ ]:
def get_chained_translation(zh_text):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a research assistant. Translate this Mandarin transcript to English. Preserve all emotional tags like <|HAPPY|> or <|LAUGHTER|>."},
            {"role": "user", "content": zh_text}
        ]
    )
    return response.choices[0].message.content

def get_direct_translation(audio_path):
    audio, orig_freq = torchaudio.load(audio_path)
    if orig_freq != 16000:
        audio = torchaudio.functional.resample(audio, orig_freq=orig_freq, new_freq=16000)
    inputs = processor(audios=audio, return_tensors="pt").to(DEVICE)
    output_tokens = direct_model.generate(**inputs, tgt_lang="eng")
    return processor.decode(output_tokens[0].tolist()[0], skip_special_tokens=True)

# The Batch Evaluator Execution

In [ ]:
def run_full_evaluation():
    root = Tk(); root.withdraw(); root.attributes('-topmost', True)
    input_dir = filedialog.askdirectory(title="Select Source Audio Folder")
    if not input_dir: return

    results = []
    extensions = ('.wav', '.mp3', '.m4a', '.mp4')
    files = [os.path.join(r, f) for r, _, fs in os.walk(input_dir) for f in fs if f.lower().endswith(extensions)]

    for audio_path in tqdm(files, desc="Processing Research Data"):
        try:
            # A. Transcribe (SenseVoice)
            res = transcription_model.generate(input=audio_path, language="zh", use_itn=True)
            zh_text = res[0]['text']
            
            # B. Diarize (Pyannote)
            diarization = diarize_pipeline(audio_path)
            speakers = ", ".join(list(set([label for _, _, label in diarization.itertracks(yield_label=True)])))

            # C. Translate (Both Methods)
            chained_en = get_chained_translation(zh_text)
            direct_en = get_direct_translation(audio_path)

            results.append({
                "File": os.path.basename(audio_path),
                "Speakers": speakers,
                "Original Mandarin": zh_text,
                "Chained English (Preserves Tags)": chained_en,
                "Direct English (Seamless)": direct_en
            })
            
            # Memory Cleanup
            torch.cuda.empty_cache(); gc.collect()
        except Exception as e:
            print(f"Error on {audio_path}: {e}")

    # Export
    df = pd.DataFrame(results)
    df.to_excel("NGSS_Translation_Evaluation.xlsx", index=False)
    print("Batch Complete. Check 'NGSS_Translation_Evaluation.xlsx'")

run_full_evaluation()

# Visualization & Final Export

In [ ]:
import matplotlib.pyplot as plt

def visualize_and_finalize(excel_path="NGSS_Translation_Evaluation.xlsx"):
    # 1. Load the results we just created
    df = pd.read_excel(excel_path)
    
    # 2. Calculate word counts for comparison
    df['Chained_Word_Count'] = df['Chained English (Preserves Tags)'].apply(lambda x: len(str(x).split()))
    df['Direct_Word_Count'] = df['Direct English (Seamless)'].apply(lambda x: len(str(x).split()))
    
    # 3. Create the Visualization
    plt.figure(figsize=(12, 6))
    x = range(len(df))
    width = 0.35
    
    plt.bar(x, df['Chained_Word_Count'], width, label='Chained (SenseVoice+GPT)', color='#4A90E2')
    plt.bar([i + width for i in x], df['Direct_Word_Count'], width, label='Direct (SeamlessM4T)', color='#50E3C2')
    
    plt.xlabel('Audio Files (Index)')
    plt.ylabel('Word Count')
    plt.title('Translation Verbosity Comparison: Chained vs. Direct')
    plt.xticks([i + width/2 for i in x], df.index)
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Save the chart for your research folder
    plt.savefig("Translation_Comparison_Chart.png")
    plt.show()
    
    print(f"Visual analysis complete. Summary table:\n")
    print(df[['File', 'Chained_Word_Count', 'Direct_Word_Count']].describe())

# Run the visualization
# visualize_and_finalize()